# Training a spam classifier

The SMS data have now been prepared for building a classifier. Specifically, this is what you have done:

- removed numbers and punctuation
- split the messages into words (or "tokens")
- removed stop words
- applied the hashing trick and
- converted to a TF-IDF representation.

Next you'll need to split the TF-IDF data into training and testing sets. Then you'll use the training data to fit a Logistic Regression model and finally evaluate the performance of that model on the testing data.

The data are stored in `sms` and `LogisticRegression` has been imported for you.

## Instructions

- Split the data into training and testing sets in a 4:1 ratio. Set the random number seed to 13 to ensure repeatability.
- Create a `LogisticRegression` object and fit it to the training data.
- Generate predictions on the testing data.
- Use the predictions to form a confusion matrix.

In [1]:
# # Import the SparkSession class
# import pyspark
# from pyspark.sql import SparkSession

# spark = SparkSession.builder.appName('sms_manipulate_columns').getOrCreate()


In [ ]:
# Intialization
import os
import sys

os.environ["SPARK_HOME"] = "/home/talentum/spark"
os.environ["PYLIB"] = os.environ["SPARK_HOME"] + "/python/lib"
# In below two lines, use /usr/bin/python2.7 if you want to use Python 2
os.environ["PYSPARK_PYTHON"] = "/usr/bin/python3.6" 
os.environ["PYSPARK_DRIVER_PYTHON"] = "/usr/bin/python3"
sys.path.insert(0, os.environ["PYLIB"] +"/py4j-0.10.7-src.zip")
sys.path.insert(0, os.environ["PYLIB"] +"/pyspark.zip")

# NOTE: Whichever package you want mention here.
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0 pyspark-shell' 
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-avro_2.11:2.4.0 pyspark-shell'
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0,org.apache.spark:spark-avro_2.11:2.4.3 pyspark-shell'
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0,org.apache.spark:spark-avro_2.11:2.4.0 pyspark-shell'

In [ ]:
#Entrypoint 2.x
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Spark SQL basic example").enableHiveSupport().getOrCreate()

# On yarn:
# spark = SparkSession.builder.appName("Spark SQL basic example").enableHiveSupport().master("yarn").getOrCreate()
# specify .master("yarn")

sc = spark.sparkContext

In [6]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

schema = StructType([
    StructField("id", IntegerType()),
	StructField("text", StringType()),
	StructField("label", IntegerType())
])
	
sms = spark.read.csv("file:///home/talentum/test-jupyter/c5-MLWithPySpark/M2-Classification/4_TurningTextIntoTables/dataset/sms.csv", 
                     sep=';', header=False, schema=schema)

from pyspark.sql.functions import regexp_replace
from pyspark.ml.feature import Tokenizer

sms = sms.withColumn('text', regexp_replace(sms.text, '[_():;,.!?\\\\\\\\-]', ' '))
sms = sms.withColumn('text', regexp_replace(sms.text, '[0-9]', ' '))
sms = sms.withColumn('text', regexp_replace(sms.text, '(^ +| +$)', ''))
sms = sms.withColumn('text', regexp_replace(sms.text, ' +', ' '))

sms = Tokenizer(inputCol='text', outputCol='words').transform(sms)
sms = sms.select('id', 'words', 'label')

from pyspark.ml.feature import StopWordsRemover, HashingTF, IDF

sms = StopWordsRemover(inputCol='words', outputCol='terms')\
.transform(sms)

sms = HashingTF(inputCol='terms', outputCol='hash', numFeatures=1024)\
.transform(sms)

sms = IDF(inputCol='hash', outputCol='features')\
.fit(sms).transform(sms)

print("Selected columns from first few rows of the sms DataFrame:")
sms.select('label', 'features').show(5)

from pyspark.ml.classification import LogisticRegression

Selected columns from first few rows of the sms DataFrame:
+-----+--------------------+
|label|            features|
+-----+--------------------+
|    0|(1024,[138,384,57...|
|    0|(1024,[215,233,27...|
|    1|(1024,[133,138],[...|
|    1|(1024,[31,47,62,3...|
|    0|(1024,[12,171,191...|
+-----+--------------------+
only showing top 5 rows



In [ ]:
# Split the data into training and testing sets
sms_train, sms_test = sms.____(____, ____)

# Fit a Logistic Regression model to the training data
logistic = ____(regParam=0.2).____(____)

# Make predictions on the testing data
prediction = logistic.____(____)

# Create a confusion matrix, comparing predictions to known labels
prediction.groupBy(____, ____).____().____()